# Data Preparation Pipeline
## Pneumonia Detection Project — Group 4

This notebook performs the full data preparation pipeline for our pneumonia detection project, which combines two datasets:
- **Chest X-Ray Images (Pneumonia)** — 5,856 JPEG images from Kaggle
- **RSNA Pneumonia Detection Challenge** — 27,901 DICOM images from Kaggle

### Install Dependencies

We install the required AWS libraries:
- `sagemaker` — AWS SageMaker SDK for managing ML workflows
- `pyathena` — Python client for querying data with Amazon Athena
- `awswrangler` — simplifies interaction between Pandas and AWS services
- `boto3` — AWS SDK for Python to interact with S3, IAM, and other services

### Verify Data in S3 Data Lake

Our raw image data is stored in a shared S3 bucket (`pneumonia-data-set-group-4`). This bucket serves as our **data lake** — a centralized repository for all raw project data.

Here we connect to the bucket and verify the folder structure and file counts across our training, test, and validation splits for both NORMAL and PNEUMONIA classes.


### Analyze File Formats

Since our project combines two different datasets, the images come in two formats:
- **DICOM (.dcm)** — medical imaging format from the RSNA dataset
- **JPEG (.jpeg)** — standard image format from the Chest X-Ray dataset

We scan all files in S3 to understand the distribution of file types. This is important because each format requires different preprocessing code to read and convert into arrays for model training.

In [1]:
%pip uninstall sagemaker -y
%pip install "sagemaker>=2.0,<3.0" -q

Found existing installation: sagemaker 2.257.3


Uninstalling sagemaker-2.257.3:


  Successfully uninstalled sagemaker-2.257.3


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install "sagemaker<3" pyathena awswrangler  --quiet
!pip install 'boto3>1.17.21' -q

In [3]:
import boto3

s3 = boto3.client("s3")
bucket = "pneumonia-data-set-group-4"

# Check what top-level folders exist
response = s3.list_objects_v2(Bucket=bucket, Prefix="raw-images/", Delimiter="/")
print("Top folders:")
for prefix in response.get("CommonPrefixes", []):
    print(f"  {prefix['Prefix']}")

# Count files in each folder
paginator = s3.get_paginator("list_objects_v2")
for split in ["train", "test", "val"]:
    for label in ["NORMAL", "PNEUMONIA"]:
        prefix = f"raw-images/{split}/{label}/"
        count = 0
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
            count += len(page.get("Contents", []))
        if count > 0:
            print(f"  {split}/{label}: {count} files")

Top folders:
  raw-images/chest-xray/
  raw-images/rsna-pneumonia/


In [4]:
import boto3

s3 = boto3.client("s3")
bucket = "pneumonia-data-set-group-4"
paginator = s3.get_paginator("list_objects_v2")

dcm_count = 0
jpeg_count = 0
other_count = 0
dcm_keys = []
for page in paginator.paginate(Bucket=bucket, Prefix="raw-images/"):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key.endswith(".dcm"):
            dcm_count += 1
            dcm_keys.append(key)
        elif key.endswith((".jpeg", ".jpg", ".png")):
            jpeg_count += 1
        else:
            other_count += 1

print(f"DICOM (.dcm):  {dcm_count}")
print(f"JPEG (.jpeg):  {jpeg_count}")
print(f"Other:         {other_count}")
print(f"Total:         {dcm_count + jpeg_count + other_count}")

DICOM (.dcm):  29684
JPEG (.jpeg):  5856
Other:         4
Total:         35544


In [5]:
dcm_keys[0]

'raw-images/rsna-pneumonia/stage_2_test_images/0000a175-0e68-4ca4-b1af-167204a7e0bc.dcm'

### Step 4: Set Up Athena Tables for Cataloging & Querying

Amazon Athena is a serverless query service that lets us run SQL queries directly on data stored in S3. Instead of looping through S3 files every time we need information, we can query our dataset using standard SQL.

Since Athena cannot query raw image files, we create a **metadata table** — a CSV file where each row represents one image with its S3 path, label (NORMAL/PNEUMONIA), split (train/test/val), source (chest_xray/rsna), and file size.

**What happens in this section:**
1. Connect to Athena using PyAthena with a staging directory for query results
2. Create a new database (`pneumonia_db`) in the AWS Glue Data Catalog
3. Scan all 33,757 images in S3 and build a metadata DataFrame
4. Upload the metadata CSV to S3
5. Create an external Athena table pointing to the CSV
6. Run SQL queries to verify the table and explore class distributions



In [6]:
import boto3
import sagemaker
import pandas as pd
from pyathena import connect

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [7]:
#Setup Connection



sess = sagemaker.Session()
default_bucket = sess.default_bucket()
region = boto3.Session().region_name
bucket = "pneumonia-data-set-group-4"

# Athena staging directory (uses YOUR default bucket for query results)
s3_staging_dir = f"s3://{default_bucket}/athena/staging"

# Connect to Athena
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

print(f"Region: {region}")
print(f"Data bucket: {bucket}")
print(f"Staging dir: {s3_staging_dir}")

Region: us-east-1
Data bucket: pneumonia-data-set-group-4
Staging dir: s3://sagemaker-us-east-1-455131748909/athena/staging


In [8]:
#Create the database

database_name = "pneumonia_db"

statement = f"CREATE DATABASE IF NOT EXISTS {database_name}"
print(statement)
pd.read_sql(statement, conn)
print("✅ Database created!")

CREATE DATABASE IF NOT EXISTS pneumonia_db


/tmp/ipykernel_68983/1516474353.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement, conn)


✅ Database created!


In [9]:
df_databases = pd.read_sql("SHOW DATABASES", conn)
print(df_databases)

if database_name in df_databases.values:
    print(f"\n✅ Database '{database_name}' exists!")
else:
    print(f"\n❌ Database '{database_name}' not found")

/tmp/ipykernel_68983/2588296401.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_databases = pd.read_sql("SHOW DATABASES", conn)


  database_name
0       default
1  pneumonia_db

✅ Database 'pneumonia_db' exists!


In [17]:
rsna_df_1 = pd.read_csv('stage_2_train_labels.csv')
rsna_df_2 = pd.read_csv('stage_2_sample_submission.csv')

In [18]:
rsna_df = pd.concat([rsna_df_1, rsna_df_2])

In [19]:
rsna_df.head()

,patientId,x,y,width,height,Target,PredictionString
0,0004cfab-14fd-4e49-80ba-63a80b6bddd6,NaN,NaN,NaN,NaN,0.0,NaN
1,00313ee0-9eaa-42f4-b0ab-c148ed3241cd,NaN,NaN,NaN,NaN,0.0,NaN
2,00322d4d-1c29-4943-afc9-b6754be640eb,NaN,NaN,NaN,NaN,0.0,NaN
3,003d8fa0-6bf1-40ed-b54c-ac657f8495c5,NaN,NaN,NaN,NaN,0.0,NaN
4,00436515-870c-4b36-a041-de91049b9ab4,264.0,152.0,213.0,379.0,1.0,NaN


In [20]:
len(rsna_df)

33227

In [23]:
rsna_df.Target.unique()

array([ 0.,  1., nan])

In [25]:
rsna_df[rsna_df.Target.isnull()]

,patientId,x,y,width,height,Target,PredictionString
0,0000a175-0e68-4ca4-b1af-167204a7e0bc,NaN,NaN,NaN,NaN,NaN,0.5 0 0 100 100
1,0005d3cc-3c3f-40b9-93c3-46231c3eb813,NaN,NaN,NaN,NaN,NaN,0.5 0 0 100 100
2,000686d7-f4fc-448d-97a0-44fa9c5d3aa6,NaN,NaN,NaN,NaN,NaN,0.5 0 0 100 100
3,000e3a7d-c0ca-4349-bb26-5af2d8993c3d,NaN,NaN,NaN,NaN,NaN,0.5 0 0 100 100
4,00100a24-854d-423d-a092-edcf6179e061,NaN,NaN,NaN,NaN,NaN,0.5 0 0 100 100
...,...,...,...,...,...,...,...
2995,c1e88810-9e4e-4f39-9306-8d314bfc1ff1,NaN,NaN,NaN,NaN,NaN,0.5 0 0 100 100
2996,c1ec035b-377b-416c-a281-f868b7c9b6c3,NaN,NaN,NaN,NaN,NaN,0.5 0 0 100 100
2997,c1ef5b66-0fd7-49d1-ae6b-5af84929414b,NaN,NaN,NaN,NaN,NaN,0.5 0 0 100 100
2998,c1ef6724-f95f-40f1-b25b-de806d9bc39d,NaN,NaN,NaN,NaN,NaN,0.5 0 0 100 100


In [21]:
rsna_label_map = {}
for dic in rsna_df.to_dict(orient='records'):
    
    rsna_label_map[dic['patientId']] = dic['Target']

In [22]:
len(rsna_label_map)

29684

In [43]:
# Build metadata from S3 images
s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")
rows = []



prefix = f"raw-images/"
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        file_name = key.split("/")[-1]
        file_type = file_name.split('.')[-1]
        if file_type in ['jpeg', 'dcm']:
            source = "rsna" if 'rsna' in key else "chest_xray"

            if source == 'rsna':
                file_name_no_stem = file_name.split('.')[0]
                label = rsna_label_map[file_name_no_stem]
            elif source == 'chest_xray':
                if 'NORMAL' in key:
                    label = 0
                else:
                    label = 1
        
            rows.append({
                "image_id": file_name.replace(".dcm", "").replace(".jpeg", "").replace(".jpg", ""),
                "s3_key": key,
                "file_name": file_name,
                "file_type": file_type,
                'label': label,
                "source": source,
                "file_size": obj["Size"]
            })

df_metadata = pd.DataFrame(rows)
print(f"Total images: {len(df_metadata)}")
df_metadata.head()

Total images: 35540


,image_id,s3_key,file_name,file_type,label,source,file_size
0,IM-0001-0001,raw-images/chest-xray/test/NORMAL/IM-0001-0001...,IM-0001-0001.jpeg,jpeg,0.0,chest_xray,252680
1,IM-0003-0001,raw-images/chest-xray/test/NORMAL/IM-0003-0001...,IM-0003-0001.jpeg,jpeg,0.0,chest_xray,329189
2,IM-0005-0001,raw-images/chest-xray/test/NORMAL/IM-0005-0001...,IM-0005-0001.jpeg,jpeg,0.0,chest_xray,408620
3,IM-0006-0001,raw-images/chest-xray/test/NORMAL/IM-0006-0001...,IM-0006-0001.jpeg,jpeg,0.0,chest_xray,252275
4,IM-0007-0001,raw-images/chest-xray/test/NORMAL/IM-0007-0001...,IM-0007-0001.jpeg,jpeg,0.0,chest_xray,408508


In [44]:
df_metadata.label.unique()

array([ 0.,  1., nan])

In [45]:
# Remove null labels
df_metadata = df_metadata[~df_metadata.label.isnull()]

print(f"Total images after cleanup: {len(df_metadata)}")
df_metadata.head()

Total images after cleanup: 32540


,image_id,s3_key,file_name,file_type,label,source,file_size
0,IM-0001-0001,raw-images/chest-xray/test/NORMAL/IM-0001-0001...,IM-0001-0001.jpeg,jpeg,0.0,chest_xray,252680
1,IM-0003-0001,raw-images/chest-xray/test/NORMAL/IM-0003-0001...,IM-0003-0001.jpeg,jpeg,0.0,chest_xray,329189
2,IM-0005-0001,raw-images/chest-xray/test/NORMAL/IM-0005-0001...,IM-0005-0001.jpeg,jpeg,0.0,chest_xray,408620
3,IM-0006-0001,raw-images/chest-xray/test/NORMAL/IM-0006-0001...,IM-0006-0001.jpeg,jpeg,0.0,chest_xray,252275
4,IM-0007-0001,raw-images/chest-xray/test/NORMAL/IM-0007-0001...,IM-0007-0001.jpeg,jpeg,0.0,chest_xray,408508


In [46]:
label_map = { 0: "NORMAL",
             1: "PNEUMONIA"
            }
df_metadata['label_int'] = df_metadata.label.astype(int)
df_metadata['label'] = df_metadata.label_int.map(label_map)

In [47]:
df_metadata.label_int.unique()

array([0, 1])

In [48]:
df_metadata['is_preprocessed'] = False
df_metadata['file_size'] = df_metadata.file_size.astype(int)

In [49]:
df_metadata.head(2)

,image_id,s3_key,file_name,file_type,label,source,file_size,label_int,is_preprocessed
0,IM-0001-0001,raw-images/chest-xray/test/NORMAL/IM-0001-0001...,IM-0001-0001.jpeg,jpeg,NORMAL,chest_xray,252680,0,False
1,IM-0003-0001,raw-images/chest-xray/test/NORMAL/IM-0003-0001...,IM-0003-0001.jpeg,jpeg,NORMAL,chest_xray,329189,0,False


In [63]:
df_metadata.s3_key.iloc[0]

'raw-images/chest-xray/test/NORMAL/IM-0001-0001.jpeg'

In [50]:
IS_DATA_OWNER = True

if IS_DATA_OWNER:
    # Save locally then upload
    csv_path = "image_metadata.csv"
    df_metadata.to_csv(csv_path, index=False)
    # Upload to YOUR default bucket (you have write access)
    s3.upload_file(csv_path, bucket, f"pneumonia-project/metadata/{csv_path}")
    print(f"Metadata uploaded to s3://{bucket}/pneumonia-project/metadata/{csv_path}")

Metadata uploaded to s3://pneumonia-data-set-group-4/pneumonia-project/metadata/image_metadata.csv


In [56]:
statement = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.image_metadata (
    image_id     STRING,
    s3_key       STRING,
    file_name    STRING,
    file_type    STRING,
    label        STRING,
    source       STRING,
    file_size    BIGINT,
    label_int    TINYINT,
    is_preprocessed    BOOLEAN
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LOCATION 's3://{bucket}/pneumonia-project/metadata/'
TBLPROPERTIES ('skip.header.line.count'='1')
"""

print(statement)
pd.read_sql(statement, conn)
print("✅ Table created!")


CREATE EXTERNAL TABLE IF NOT EXISTS pneumonia_db.image_metadata (
    image_id     STRING,
    s3_key       STRING,
    file_name    STRING,
    file_type    STRING,
    label        STRING,
    source       STRING,
    file_size    BIGINT,
    label_int    TINYINT,
    is_preprocessed    BOOLEAN
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LOCATION 's3://pneumonia-data-set-group-4/pneumonia-project/metadata/'
TBLPROPERTIES ('skip.header.line.count'='1')



/tmp/ipykernel_68983/665401036.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement, conn)


✅ Table created!


In [57]:
df_tables = pd.read_sql(f"SHOW TABLES IN {database_name}", conn)
print(df_tables)

if "image_metadata" in df_tables.values:
    print("\n✅ Table 'image_metadata' exists!")
else:
    print("\n❌ Table not found")

/tmp/ipykernel_68983/1759804955.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tables = pd.read_sql(f"SHOW TABLES IN {database_name}", conn)


         tab_name
0  image_metadata

✅ Table 'image_metadata' exists!


In [59]:
# Query 1 - Count all images
df_count = pd.read_sql(f"""
    SELECT COUNT(*) as total_images 
    FROM {database_name}.image_metadata
""", conn)
print("Total images:")
print(df_count)

# Query 2 - Class distribution
df_labels = pd.read_sql(f"""
    SELECT label, COUNT(*) as count 
    FROM {database_name}.image_metadata 
    GROUP BY label
""", conn)
print("\nClass distribution:")
print(df_labels)

# Query 3 - Breakdown by split, label, source
df_breakdown = pd.read_sql(f"""
    SELECT label, source, COUNT(*) as count 
    FROM {database_name}.image_metadata 
    GROUP BY label, source
    ORDER BY label, source
""", conn)
print("\nFull breakdown:")
print(df_breakdown)

/tmp/ipykernel_68983/3106592038.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_count = pd.read_sql(f"""


Total images:
   total_images
0         32540


/tmp/ipykernel_68983/3106592038.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_labels = pd.read_sql(f"""



Class distribution:
       label  count
0  PNEUMONIA  10285
1     NORMAL  22255


/tmp/ipykernel_68983/3106592038.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_breakdown = pd.read_sql(f"""



Full breakdown:
       label      source  count
0     NORMAL  chest_xray   1583
1     NORMAL        rsna  20672
2  PNEUMONIA  chest_xray   4273
3  PNEUMONIA        rsna   6012


### Step 5: Exploratory Data Analysis (EDA)

In this section, we explore our combined pneumonia dataset to understand its characteristics before building a model. EDA helps us identify potential issues like class imbalance, inconsistent image sizes, and differences between our two data sources (Chest X-Ray and RSNA).

**What we analyze:**
1. **Class distribution** — How balanced are NORMAL vs PNEUMONIA labels across splits and sources?
2. **Sample images** — Visual comparison of Normal vs Pneumonia X-rays from both datasets
3. **Image dimensions** — Are image sizes consistent? Do they need resizing?
4. **Pixel intensity** — Are there brightness/contrast differences between classes or sources?

These insights will guide our preprocessing and feature engineering decisions in the next steps.

In [5]:
!pip install pydicom opencv-python-headless matplotlib seaborn --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
skops 0.14.0 requires prettytable>=3.9, which is not installed.
amazon-sagemaker-sql-magic 0.1.4 requires numpy<2, but you have numpy 2.4.6 which is incompatible.
autogluon-common 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.6 which is incompatible.
autogluon-common 1.5.0 requires pyarrow<21.0.0,>=7.0.0, but you have pyarrow 21.0.0 which is incompatible.
autogluon-core 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.6 which is incompatible.
autogluon-featu

In [6]:
import boto3
import numpy as np
import cv2
import pydicom
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import io
import random

s3 = boto3.client("s3")
bucket = "pneumonia-data-set-group-4"

print("✅ Setup complete")

✅ Setup complete


In [ ]:
# this code is repeated from up further up? can this be deleted? -sc
# Build metadata from S3 
paginator = s3.get_paginator("list_objects_v2")
rows = []

for split in ["train", "test", "val"]:
    for label in ["NORMAL", "PNEUMONIA"]:
        prefix = f"raw-images/{split}/{label}/"
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
            for obj in page.get("Contents", []):
                key = obj["Key"]
                if ".ipynb_checkpoint" in key:
                    continue  # skip junk files
                file_name = key.split("/")[-1]
                file_type = "dcm" if file_name.endswith(".dcm") else "jpeg"
                source = "rsna" if file_type == "dcm" else "chest_xray"

                rows.append({
                    "image_id": file_name.replace(".dcm", "").replace(".jpeg", "").replace(".jpg", ""),
                    "s3_key": key,
                    "file_name": file_name,
                    "split": split,
                    "label": label,
                    "file_type": file_type,
                    "source": source,
                    "file_size": obj["Size"]
                })

df = pd.DataFrame(rows)
print(f"Total images: {len(df)}")
df.head()

In [ ]:
#Dataset overview

print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)

print(f"\nTotal images: {len(df)}")

print(f"\nBy label:")
print(df["label"].value_counts())

print(f"\nBy split:")
print(df["split"].value_counts())

print(f"\nBy source:")
print(df["source"].value_counts())

normal = len(df[df["label"] == "NORMAL"])
pneumonia = len(df[df["label"] == "PNEUMONIA"])
print(f"\nClass ratio - Normal:Pneumonia = {normal/pneumonia:.2f}:1")

In [ ]:
#Class distribution charts

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall class balance
counts = df["label"].value_counts()
axes[0].bar(counts.index, counts.values, color=["steelblue", "salmon"])
axes[0].set_title("Overall Class Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, str(v), ha="center", fontweight="bold")

# By split
split_counts = df.groupby(["split", "label"]).size().unstack(fill_value=0)
split_counts.plot(kind="bar", ax=axes[1], color=["steelblue", "salmon"])
axes[1].set_title("Class Distribution by Split")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Label")

# By source
source_counts = df.groupby(["source", "label"]).size().unstack(fill_value=0)
source_counts.plot(kind="bar", ax=axes[2], color=["steelblue", "salmon"])
axes[2].set_title("Class Distribution by Source")
axes[2].set_ylabel("Count")
axes[2].tick_params(axis="x", rotation=0)
axes[2].legend(title="Label")

plt.tight_layout()
plt.show()

In [ ]:
#helper function to load images from S3

def load_image_from_s3(s3_key):
    """Load an image from S3 - handles both JPEG and DICOM formats"""
    response = s3.get_object(Bucket=bucket, Key=s3_key)
    img_bytes = response["Body"].read()

    if s3_key.endswith(".dcm"):
        # DICOM format (RSNA dataset)
        ds = pydicom.dcmread(io.BytesIO(img_bytes))
        img = ds.pixel_array
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    else:
        # JPEG format (Chest X-Ray dataset)
        img_array = np.frombuffer(img_bytes, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)

    return img

# Test it
test_key = df.iloc[0]["s3_key"]
test_img = load_image_from_s3(test_key)
print(f"✅ Loaded image — shape: {test_img.shape}, dtype: {test_img.dtype}")

In [ ]:
#Visualize sample images:

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

normal_samples = df[df["label"] == "NORMAL"].sample(5, random_state=42)
pneumonia_samples = df[df["label"] == "PNEUMONIA"].sample(5, random_state=42)

# Normal images (top row)
for i, (_, row) in enumerate(normal_samples.iterrows()):
    img = load_image_from_s3(row["s3_key"])
    axes[0][i].imshow(img, cmap="gray")
    axes[0][i].set_title(f"NORMAL\n{row['source']}\n{img.shape}", fontsize=9)
    axes[0][i].axis("off")

# Pneumonia images (bottom row)
for i, (_, row) in enumerate(pneumonia_samples.iterrows()):
    img = load_image_from_s3(row["s3_key"])
    axes[1][i].imshow(img, cmap="gray")
    axes[1][i].set_title(f"PNEUMONIA\n{row['source']}\n{img.shape}", fontsize=9)
    axes[1][i].axis("off")

plt.suptitle("Sample Images: Normal vs Pneumonia", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
#Image size distribution
# Sample 500 images to check sizes

sample_df = df.sample(min(500, len(df)), random_state=42)

heights, widths, sources, labels = [], [], [], []

print("Analyzing image sizes ")
for _, row in sample_df.iterrows():
    try:
        img = load_image_from_s3(row["s3_key"])
        heights.append(img.shape[0])
        widths.append(img.shape[1])
        sources.append(row["source"])
        labels.append(row["label"])
    except:
        continue

size_df = pd.DataFrame({"height": heights, "width": widths, "source": sources, "label": labels})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(data=size_df, x="height", hue="source", ax=axes[0], bins=30)
axes[0].set_title("Image Height Distribution")

sns.histplot(data=size_df, x="width", hue="source", ax=axes[1], bins=30)
axes[1].set_title("Image Width Distribution")

sns.scatterplot(data=size_df, x="width", y="height", hue="source", alpha=0.5, ax=axes[2])
axes[2].set_title("Height vs Width")

plt.tight_layout()
plt.show()

print("\nImage size stats:")
print(size_df.groupby("source")[["height", "width"]].describe())

In [ ]:
#Pixel intensity analysis

sample_normal = df[df["label"] == "NORMAL"].sample(50, random_state=42)
sample_pneumonia = df[df["label"] == "PNEUMONIA"].sample(50, random_state=42)

normal_means, pneumonia_means = [], []
normal_stds, pneumonia_stds = [], []

print("Analyzing pixel intensities...")

for _, row in sample_normal.iterrows():
    try:
        img = load_image_from_s3(row["s3_key"])
        normal_means.append(img.mean())
        normal_stds.append(img.std())
    except:
        continue

for _, row in sample_pneumonia.iterrows():
    try:
        img = load_image_from_s3(row["s3_key"])
        pneumonia_means.append(img.mean())
        pneumonia_stds.append(img.std())
    except:
        continue

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(normal_means, bins=20, alpha=0.6, label="Normal", color="steelblue")
axes[0].hist(pneumonia_means, bins=20, alpha=0.6, label="Pneumonia", color="salmon")
axes[0].set_title("Mean Pixel Intensity Distribution")
axes[0].set_xlabel("Mean Pixel Value")
axes[0].legend()

axes[1].hist(normal_stds, bins=20, alpha=0.6, label="Normal", color="steelblue")
axes[1].hist(pneumonia_stds, bins=20, alpha=0.6, label="Pneumonia", color="salmon")
axes[1].set_title("Pixel Intensity Std Distribution")
axes[1].set_xlabel("Std Pixel Value")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Normal    — Mean: {np.mean(normal_means):.1f}, Std: {np.mean(normal_stds):.1f}")
print(f"Pneumonia — Mean: {np.mean(pneumonia_means):.1f}, Std: {np.mean(pneumonia_stds):.1f}")

In [ ]:
print("=" * 50)
print("EDA SUMMARY")
print("=" * 50)
print(f"""
Total Images:     {len(df)}
  - Normal:       {len(df[df['label']=='NORMAL'])} ({len(df[df['label']=='NORMAL'])/len(df)*100:.1f}%)
  - Pneumonia:    {len(df[df['label']=='PNEUMONIA'])} ({len(df[df['label']=='PNEUMONIA'])/len(df)*100:.1f}%)

Sources:
  - Chest X-Ray:  {len(df[df['source']=='chest_xray'])} JPEG images
  - RSNA:         {len(df[df['source']=='rsna'])} DICOM images

Splits:
  - Train:        {len(df[df['split']=='train'])}
  - Test:         {len(df[df['split']=='test'])}
  - Val:          {len(df[df['split']=='val'])}

Key Findings:
  1. Class imbalance: {len(df[df['label']=='NORMAL'])/len(df[df['label']=='PNEUMONIA']):.2f}:1 ratio (Normal:Pneumonia)
  2. Two file formats need unified preprocessing (DICOM + JPEG)
  3. Image sizes vary — need resizing to 512x512 (per config.py)
""")

### Step 6: Feature Engineering & SageMaker Feature Store

In this section, we preprocess our raw X-ray images to make them ready for CNN model training, and store a **training manifest** in Amazon SageMaker Feature Store.

**Image Preprocessing Pipeline:**
1. Read image from S3 (handles both DICOM and JPEG formats)
2. Normalize pixel values to 0-255 (uint8)
3. Apply CLAHE contrast enhancement (improves X-ray detail visibility)
4. Resize to 512×512 (uniform size for CNN input)
5. Save preprocessed image back to S3

**Feature Store Manifest:**

Since we are using a CNN for classification, the model learns its own features directly from the images — we don't need handcrafted features like edge density or skewness. Instead, we store a **training manifest** in the Feature Store — a queryable lookup table that maps each image to its preprocessed S3 path, label, and normalization statistics.

This allows the training pipeline to query the Feature Store and immediately know:
- Where each preprocessed image lives in S3
- What label it has (0 = Normal, 1 = Pneumonia)
- Which split it belongs to (train/test/val/production)
- The pixel mean and std for normalization during training

This eliminates the need to scan S3 folders and guess labels from directory names every time we train.

In [60]:
!pip install pydicom opencv-python-headless --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
skops 0.14.0 requires prettytable>=3.9, which is not installed.
amazon-sagemaker-sql-magic 0.1.4 requires numpy<2, but you have numpy 2.4.6 which is incompatible.
autogluon-common 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.6 which is incompatible.
autogluon-common 1.5.0 requires pyarrow<21.0.0,>=7.0.0, but you have pyarrow 21.0.0 which is incompatible.
autogluon-core 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.6 which is incompatible.
autogluon-featu

In [61]:
# Setup

import boto3
import sagemaker
import pandas as pd
import numpy as np
import cv2
import pydicom
import io
import time
import tempfile
import os
from datetime import datetime
from sagemaker.feature_store.feature_group import FeatureGroup

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
default_bucket = sess.default_bucket()
s3 = boto3.client("s3")
bucket = "pneumonia-data-set-group-4"

print(f"Role: {role}")
print(f"Region: {region}")
print(f"Default bucket: {default_bucket}")

Role: arn:aws:iam::455131748909:role/LabRole
Region: us-east-1
Default bucket: sagemaker-us-east-1-455131748909


In [64]:
# helper functions to load images from S3 and preprocess them for CNN training

IMG_SIZE = (512, 512)

def load_image_from_s3(s3_key):
    """Load image from S3 - handles both JPEG and DICOM"""
    response = s3.get_object(Bucket=bucket, Key=s3_key)
    img_bytes = response["Body"].read()

    if s3_key.endswith(".dcm"):
        ds = pydicom.dcmread(io.BytesIO(img_bytes))
        img = ds.pixel_array
    else:
        img_array = np.frombuffer(img_bytes, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)

    return img


def preprocess_image(img):
    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)

    img = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_LINEAR)

    return img


# Test it
test_key = 'raw-images/chest-xray/test/NORMAL/IM-0001-0001.jpeg'
test_img = load_image_from_s3(test_key)
processed = preprocess_image(test_img)
print(f"Raw: {test_img.shape} → Preprocessed: {processed.shape}")

Raw: (1317, 1857) → Preprocessed: (512, 512)


In [65]:
#Get metadata from Athena

import awswrangler as wr

df_meta = wr.athena.read_sql_query(
    "SELECT * FROM pneumonia_db.image_metadata",
    database="pneumonia_db"
)

# Remove checkpoint files if any
df_meta = df_meta[~df_meta["s3_key"].str.contains(".ipynb_checkpoint")]

print(f"Total images to process: {len(df_meta)}")

2026-06-07 21:22:43,306	WARNING services.py:2137 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 1908387840 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=4.39gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.


2026-06-07 21:22:44,499	INFO worker.py:2007 -- Started a local Ray instance.


/opt/conda/lib/python3.12/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Total images to process: 32540


### upload preprocessed images to S3

In [1]:
IS_DATA_OWNER = False

In [67]:

if IS_DATA_OWNER:
    manifest_rows = []
    errors = 0
    n = len(df_meta)
    print(f"Preprocessing {n} images...")
    
    for i, (_, row) in enumerate(df_meta.iterrows()):
        try:
            # Load raw image
            img = load_image_from_s3(row["s3_key"])
    
            # Preprocess
            processed = preprocess_image(img)
    
            # Calculate pixel stats (for normalization during training)
            pixel_mean = float(np.mean(processed))
            pixel_std = float(np.std(processed))
    
            # Save preprocessed image to S3
            preprocessed_key = f"preprocessed-images/{row['label']}/{row['image_id']}.png"
            _, buf = cv2.imencode(".png", processed)
            s3.put_object(
                Bucket=bucket,
                Key=preprocessed_key,
                Body=buf.tobytes()
            )
    
            # Build manifest row
            manifest_rows.append({
                "image_id": str(row["image_id"]),
                "raw_s3_key": str(row["s3_key"]),
                "preprocessed_s3_key": preprocessed_key,
                "label": str(row["label"]),
                "label_int": int(1 if row["label"] == "PNEUMONIA" else 0),
                "source": str(row["source"]),
                "file_type": str(row["file_type"]),
                "pixel_mean": round(pixel_mean, 4),
                "pixel_std": round(pixel_std, 4),
                "img_height": 512,
                "img_width": 512,
                "event_time": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
            })
    
        except Exception as e:
            print(e)
            errors += 1
            continue
    
        if (i + 1) % 1000 == 0:
            print(f"  Processed {i + 1}/{n}...")
    
    df_manifest = pd.DataFrame(manifest_rows)
    print(f"\n✅ Done! Preprocessed {len(df_manifest)} images ({errors} errors)")
    

Preprocessing 32540 images...


/tmp/ipykernel_68983/1160455216.py:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "event_time": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")


  Processed 1000/32540...


  Processed 2000/32540...


  Processed 3000/32540...


...

  Processed 29000/32540...


  Processed 30000/32540...


  Processed 31000/32540...


  Processed 32000/32540...



✅ Done! Preprocessed 32540 images (0 errors)


In [68]:
df_manifest.head(3)

,image_id,raw_s3_key,preprocessed_s3_key,label,label_int,source,file_type,pixel_mean,pixel_std,img_height,img_width,event_time
0,IM-0001-0001,raw-images/chest-xray/test/NORMAL/IM-0001-0001...,preprocessed-images/NORMAL/IM-0001-0001.png,NORMAL,0,chest_xray,jpeg,122.9340,60.6187,512,512,2026-06-07T21:26:28Z
1,IM-0003-0001,raw-images/chest-xray/test/NORMAL/IM-0003-0001...,preprocessed-images/NORMAL/IM-0003-0001.png,NORMAL,0,chest_xray,jpeg,130.1997,60.4193,512,512,2026-06-07T21:26:28Z
2,IM-0005-0001,raw-images/chest-xray/test/NORMAL/IM-0005-0001...,preprocessed-images/NORMAL/IM-0005-0001.png,NORMAL,0,chest_xray,jpeg,128.4836,60.4571,512,512,2026-06-07T21:26:28Z


In [70]:
df_manifest.label.unique()

array(['NORMAL', 'PNEUMONIA'], dtype=object)

#### update with athena db the new pneumonia metadata file

In [71]:
if IS_DATA_OWNER:
    # Save locally then upload
    csv_path = "image_metadata.csv"
    df_manifest.to_csv(csv_path, index=False)
    # Upload to shared s3 bucket
    s3.upload_file(csv_path, bucket, f"pneumonia-project/metadata/{csv_path}")
    print(f"✅ Updated Metadata uploaded to s3://{bucket}/pneumonia-project/metadata/{csv_path}")

✅ Updated Metadata uploaded to s3://pneumonia-data-set-group-4/pneumonia-project/metadata/image_metadata.csv


#### recreate table

In [72]:
database_name = "pneumonia_db"

In [73]:
drop_statement = f"""
DROP TABLE IF EXISTS {database_name}.image_metadata
"""
pd.read_sql(drop_statement, conn)

/tmp/ipykernel_68983/2168075215.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(drop_statement, conn)


""


In [74]:
statement = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.image_metadata (
    image_id              STRING,
    raw_s3_key            STRING,
    preprocessed_s3_key   STRING,
    label                 STRING,
    label_int             TINYINT,
    source                STRING,
    file_type             STRING,
    pixel_mean            DOUBLE,
    pixel_std             DOUBLE,
    img_height            INT,
    img_width             INT,
    event_time            STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LOCATION 's3://{bucket}/pneumonia-project/metadata/'
TBLPROPERTIES ('skip.header.line.count'='1')
"""
pd.read_sql(statement, conn)
print("✅ Table created!")

/tmp/ipykernel_68983/2802754334.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement, conn)


✅ Table created!
